# 19. LangGraph 初识：多节点条件路由

## 一、LangGraph 的五个基本概念

| 概念 | 含义 | 本 Notebook 中的对应 |
|---|---|---|
| **Node（节点）** | 图中的执行单元，本质是一个**接收状态、返回状态增量的函数** | 控制节点 `router`、故事/笑话/科普节点 |
| **Starting Edge（开始边）** | 定义图从哪个节点开始运行 | `add_edge(START, 'router')` |
| **Normal Edge（普通边）** | A 执行完固定去 B | 内容节点执行完 → `END` |
| **Conditional Edge（条件边）** | A 执行完，由一个**路由函数的返回值**动态决定去哪个节点 | `add_conditional_edges('router', ...)` |
| **End（结束节点）** | 图的终点 | `END` |

贯穿所有节点的是 **State（状态）**：一个在节点之间传递的字典，每个节点返回其中一部分字段的更新，LangGraph 负责合并。

## 二、本例要搭的图

```
START → router（控制节点：判断意图）
             ├─ story    → 故事节点 → END
             ├─ joke     → 笑话节点 → END
             └─ science  → 科普节点 → END
```

**所有节点都调用同一个本地模型 `Qwen1.5-1.8B`（GPU 推理）**，区别只在提示词。

> ⚠️ 注意：本地的 `Qwen1.5-1.8B` 是 **base 基座模型**（不是 -Chat 指令微调版），本 Notebook 针对它做了三处适配：
> - **控制节点**：模型不擅长「只回答一个词」的指令，改用 **few-shot 模式补全**做分类，再加关键词规则兜底；
> - **内容节点**：用 **one-shot 示范**（在消息里放一轮示例问答）教它按体裁输出、答完即止，
>   并把 `<|im_end|>` 加入停止符；用**贪心解码**保证结果可复现，配合重复惩罚避免复读；
> - 基座模型偶尔会脑补「下一轮请求」式续写，最后再用一个小正则清洗兜底。

## 三、环境准备：加载本地 Qwen1.5-1.8B

用 transformers 直接加载（不需要 LangChain 的模型包装器，LangGraph 节点里可以调用任意 Python 函数）。

In [1]:
import re
from typing import Annotated, TypedDict
import operator

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from langgraph.graph import StateGraph, START, END

MODEL_PATH = r'S:\PythonAILearningCourse\15-transformers-basics\models\Qwen1.5-1.8B'

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# transformers 5.x 用 dtype 参数，旧版是 torch_dtype，做个兼容
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.bfloat16).to('cuda')
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.bfloat16).to('cuda')
model.eval()

# <|im_end|> 是 ChatML 的一轮结束符，基座模型默认不会把它当停止符，需要手动加
IM_END_ID = tokenizer.convert_tokens_to_ids('<|im_end|>')
STOP_IDS = [tokenizer.eos_token_id, IM_END_ID]
print('模型加载完成 →', model.device, model.dtype, '| 停止符 ids：', STOP_IDS)

S:\PythonAILearningCourse\15-transformers-basics\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 291/291 [00:00<00:00, 4011.66it/s]


模型加载完成 → cuda:0 torch.bfloat16 | 停止符 ids： [151643, 151645]


In [2]:
# 基座模型偶尔在答完后脑补「下一轮请求」（如 Human:/请帮我写…），用正则识别并截断
RAMBLE_RE = re.compile(
    r'\n\s*(?:Human[:：]|User[:：]|user\b|system\b|assistant\b|用户[:：]'
    r'|请(?:帮我|给我|把|你|以|写|讲|问))')


def llm_chat(system: str, user: str, example: tuple | None = None,
             max_new_tokens: int = 400) -> str:
    """ChatML 对话生成（内容节点用）：one-shot 示范 + 贪心解码。

    example: 可选的一轮 (示例问题, 示例回答)。基座模型不会主动停止，
             看一轮「问 → 一段回答 → <|im_end|>」的示范就能学会收尾格式。
    """
    messages = [{'role': 'system', 'content': system}]
    if example is not None:
        messages += [
            {'role': 'user', 'content': example[0]},
            {'role': 'assistant', 'content': example[1]},
        ]
    messages.append({'role': 'user', 'content': user})

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,                # 贪心解码：教学结果确定可复现
            repetition_penalty=1.1,         # 抑制基座模型的复读循环
            eos_token_id=STOP_IDS,          # 遇到 <|im_end|> 也停止
        )
    n = inputs['input_ids'].shape[1]
    text = tokenizer.decode(out[0][n:], skip_special_tokens=True).strip()

    m = RAMBLE_RE.search(text)
    return text[:m.start()].strip() if m else text


def llm_complete(prompt: str, max_new_tokens: int = 8) -> str:
    """纯文本续写（控制节点用：贪心解码，分类结果稳定可复现）"""
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
        )
    n = inputs['input_ids'].shape[1]
    return tokenizer.decode(out[0][n:], skip_special_tokens=True).strip()

## 四、控制节点的「大脑」：few-shot 分类器

给基座模型看几组「请求 → 类别」的样例，它就能对新请求做模式补全。
再用正则从输出中提取类别词，提取不到时用**关键词规则兜底**，保证图永远能路由出去。

In [3]:
# 类别键 → 中文名（仅用于打印展示）
CATEGORIES = {'story': '故事节点', 'joke': '笑话节点', 'science': '科普节点'}

# few-shot 样例：每条独立成块，格式严格统一，模型只需补全最后一个类别词
FEWSHOT = (
    '下面把用户请求分成三类，只准回答一个英文单词。\n'
    'story=想要听故事，joke=想要听笑话，science=想了解科学知识。\n\n'
    '请求：给我讲一个宇航员在月球上的故事\n类别：story\n\n'
    '请求：有没有轻松点的笑话\n类别：joke\n\n'
    '请求：黑洞为什么连光都逃不出来\n类别：science\n\n'
    '请求：我想听个睡前小故事\n类别：story\n\n'
    '请求：为什么天是蓝的\n类别：science\n\n'
    '请求：来个搞笑段子\n类别：joke\n\n'
)

# 关键词兜底规则（模型输出无法解析时启用）
KEYWORD_RULES = [
    ('joke', ['笑话', '段子', '搞笑', '逗乐', '开心一下']),
    ('science', ['为什么', '原理', '科普', '是什么', '怎么回事', '科学', '解释']),
    ('story', ['故事', '编一个', '童话', '小说', '睡前']),
]


def classify(topic: str) -> tuple[str, str]:
    """返回 (类别键, 模型原始输出)。优先 LLM，失败走关键词兜底，再不行默认 story。"""
    raw = llm_complete(FEWSHOT + f'请求：{topic}\n类别：', max_new_tokens=6)
    m = re.search(r'story|joke|science', raw.lower())
    if m:
        return m.group(0), raw
    for cat, kws in KEYWORD_RULES:
        if any(kw in topic for kw in kws):
            return cat, raw + '（关键词兜底）'
    return 'story', raw + '（默认 story）'


# 先脱离图单独验证分类器
for q in ['讲个程序员的笑话听听', '恐龙是怎么灭绝的？', '给我编一个猫咪冒险的故事']:
    cat, raw = classify(q)
    print(f'{q!r:>26} → {cat:8s} | 原始输出: {raw!r}')

              '讲个程序员的笑话听听' → joke     | 原始输出: 'joke'
               '恐龙是怎么灭绝的？' → science  | 原始输出: 'science'
            '给我编一个猫咪冒险的故事' → story    | 原始输出: 'story'


## 五、定义全局状态 State

`TypedDict` 声明状态里有哪些字段；`Annotated[list, operator.add]` 表示 `steps` 字段用**列表拼接**的方式合并——
这样每个节点只返回自己那一步的轨迹 `['joke']`，LangGraph 会自动把它们累加起来。

In [4]:
class GraphState(TypedDict):
    topic: str                                   # 用户原始请求
    category: str                                # 控制节点判定的类别 story/joke/science
    answer: str                                  # 内容节点生成的结果
    steps: Annotated[list[str], operator.add]    # 执行轨迹（reducer：列表拼接）

## 六、定义节点：每个节点就是一个函数

节点函数签名：`def node(state: GraphState) -> dict`。
**关键约定：节点必须返回「状态增量字典」，不能返回裸字符串**，否则 LangGraph 无法合并状态。

三个内容节点各带一条 one-shot 示范，教基座模型输出符合体裁且答完即止。

In [5]:
# ---------- 控制节点：判断意图，决定下一个节点 ----------
def router_node(state: GraphState) -> dict:
    cat, raw = classify(state['topic'])
    print(f'[控制节点] 原始输出={raw!r} → 路由到「{CATEGORIES[cat]}」')
    return {'category': cat, 'steps': ['router']}


# ---------- 故事节点 ----------
STORY_EXAMPLE = (
    '讲一个小动物的故事',
    '小松鼠攒了一冬天的松果，开春却全发了芽。它没有难过，把小松树苗种满山坡，'
    '几年后拥有了整片松林。',
)


def story_node(state: GraphState) -> dict:
    print('[故事节点] 正在创作短篇故事…')
    answer = llm_chat(
        '你是故事大王。根据用户主题讲一个 200 字左右的中文短篇故事，'
        '要有情节起伏，只输出故事正文，不要解释。',
        state['topic'], example=STORY_EXAMPLE)
    return {'answer': answer, 'steps': ['story']}


# ---------- 笑话节点 ----------
JOKE_EXAMPLE = (
    '讲个产品经理的笑话',
    '产品经理对程序员说：「这个需求很简单，就改一个按钮颜色。」'
    '程序员改完，整个网站都变了颜色。',
)


def joke_node(state: GraphState) -> dict:
    print('[笑话节点] 正在编笑话…')
    answer = llm_chat(
        '你是幽默大师。根据用户主题讲一个轻松健康、适合全家听的中文笑话，'
        '50~120 字，结尾有包袱，只输出笑话正文。',
        state['topic'], example=JOKE_EXAMPLE)
    return {'answer': answer, 'steps': ['joke']}


# ---------- 科普节点 ----------
SCIENCE_EXAMPLE = (
    '大海为什么是咸的',
    '海水之所以咸，是因为雨水冲刷陆地岩石，把盐分带进河流、最终汇入大海；'
    '而海水蒸发时盐分不会被带走，亿万年积累下来，海水就变咸了。',
)


def science_node(state: GraphState) -> dict:
    print('[科普节点] 正在撰写科普讲解…')
    answer = llm_chat(
        '你是科普老师。用通俗生动的中文向中学生解释用户提出的科学问题，'
        '200 字左右，可以打比方，只输出讲解正文。',
        state['topic'], example=SCIENCE_EXAMPLE)
    return {'answer': answer, 'steps': ['science']}

## 七、构图：加节点、连边（普通边 + 条件边）

`add_conditional_edges` 的第二个参数是**路由函数**：它读状态、返回一个键，
第三个参数字典把键映射到目标节点名。

In [6]:
builder = StateGraph(GraphState)

# 1) 注册节点
builder.add_node('router', router_node)
builder.add_node('story', story_node)
builder.add_node('joke', joke_node)
builder.add_node('science', science_node)

# 2) Starting Edge：图从 router 开始
builder.add_edge(START, 'router')

# 3) Conditional Edge：router 执行完，按 state['category'] 动态分流
builder.add_conditional_edges(
    'router',
    lambda state: state['category'],          # 路由函数，返回 story/joke/science
    {'story': 'story', 'joke': 'joke', 'science': 'science'},
)

# 4) Normal Edge：三个内容节点执行完都结束
builder.add_edge('story', END)
builder.add_edge('joke', END)
builder.add_edge('science', END)

# 5) 编译成可执行的图
graph = builder.compile()
print('图编译完成，节点：', list(graph.get_graph().nodes))
print()
# Mermaid 流程图文本（可复制到 https://mermaid.live 渲染）
print(graph.get_graph().draw_mermaid())

图编译完成，节点： ['__start__', 'router', 'story', 'joke', 'science', '__end__']

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router(router)
	story(story)
	joke(joke)
	science(science)
	__end__([<p>__end__</p>]):::last
	__start__ --> router;
	router -.-> joke;
	router -.-> science;
	router -.-> story;
	joke --> __end__;
	science --> __end__;
	story --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 八、运行图：invoke + 观察执行轨迹

调用 `graph.invoke(初始状态)`，观察三个不同请求如何被控制节点分发到不同分支。
想试自己的主题，改 `samples` 列表里的文本即可（贪心解码，同样输入结果可复现）。

In [7]:
samples = [
    '讲个程序员的笑话听听',
    '恐龙是怎么灭绝的？',
    '给我编一个猫咪冒险的故事',
]

for topic in samples:
    print('=' * 78)
    print('用户请求：', topic)
    result = graph.invoke({'topic': topic, 'steps': []})
    print('执行轨迹：START → ' + ' → '.join(result['steps']) + ' → END')
    print('-' * 78)
    print(result['answer'])
    print()

print('=' * 78)
print('最终完整状态：')
print({k: (v[:60] + '…') if k == 'answer' else v for k, v in result.items()})

用户请求： 讲个程序员的笑话听听
[控制节点] 原始输出='joke' → 路由到「笑话节点」
[笑话节点] 正在编笑话…


执行轨迹：START → router → joke → END
------------------------------------------------------------------------------
为什么程序员总是喜欢用“break”和“continue”？因为它们可以让他们在程序中跳过某些代码块，从而更快地完成任务。

用户请求： 恐龙是怎么灭绝的？
[控制节点] 原始输出='science' → 路由到「科普节点」
[科普节点] 正在撰写科普讲解…


执行轨迹：START → router → science → END
------------------------------------------------------------------------------
恐龙灭绝的原因至今仍是一个谜团，但有一些理论认为它们可能与气候变化有关。
1. 气候变化：一些科学家认为，气候变化可能是导致恐龙灭绝的主要原因。在6500万年前的一次大规模冰河期期间，地球上的气候发生了巨大的变化，导致许多物种灭绝。这些变化包括全球气温下降和海平面上升等。这些因素可能会对恐龙造成影响，因为它们生活在不同的环境中，并且需要适应不同的气候条件。
2. 火山爆发：火山爆发也可能对恐龙灭绝产生影响。在过去的几百万年里，地球上曾经发生过多次大规模的火山喷发，其中一些喷发可能导致大气中的二氧化碳浓度增加，从而影响到地球的气候。这种情况下，恐龙可能无法适应新的环境，因此灭绝。
3. 地震：地震也可能是导致恐龙灭绝的一个因素。在过去的几百万年里，地球经历了许多地震，其中一些可能对恐龙造成了影响。例如，在6500万年前的一次地震中，地球表面的板块移动，导致了恐龙栖息地的改变，这可能使恐龙失去了生存的机会。
总之，恐龙灭绝的原因仍然存在争议，但是气候变化、火山爆发和地震等因素都可能对它们造成影响。虽然我们目前还不能完全确定恐龙灭绝的确切原因，但我们可以通过研究化石和其他证据来了解这个过程。

用户请求： 给我编一个猫咪冒险的故事
[控制节点] 原始输出='story' → 路由到「故事节点」
[故事节点] 正在创作短篇故事…


执行轨迹：START → router → story → END
------------------------------------------------------------------------------
从前有一只名叫咪咪的小猫，它非常喜欢探险和冒险。一天，咪咪决定去寻找传说中的神秘宝藏。
在旅途中，咪咪遇到了一只叫做小花的兔子，它们一起踏上了寻找宝藏的旅程。他们走过了茂密的森林、陡峭的山峰和危险的河流，经历了许多困难和挑战。
最终，他们在一座古老的城堡里找到了宝藏。但是，这个城堡被一群邪恶的狼守护着。咪咪和小花必须想出办法才能进入城堡，并找到宝藏。
他们开始了一场激烈的战斗，但最终咪咪成功地打败了狼群并获得了宝藏。从此以后，咪咪成为了英雄，它的冒险故事也传遍了整个城市。

最终完整状态：
{'topic': '给我编一个猫咪冒险的故事', 'category': 'story', 'answer': '从前有一只名叫咪咪的小猫，它非常喜欢探险和冒险。一天，咪咪决定去寻找传说中的神秘宝藏。\n在旅途中，咪咪遇到了一只叫做小花…', 'steps': ['router', 'story']}


## 九、小结

1. **LangGraph 编程四步走**：定义 `State` → 写节点函数（返回状态增量 dict）→ 用边连图（普通边/条件边）→ `compile()` 后 `invoke()`。
2. **条件路由是核心**：控制节点本身不直接决定跳转，它只把判断结果写进状态；真正的跳转由 `add_conditional_edges` 的路由函数完成，节点逻辑与路由逻辑解耦。
3. **State 的 reducer 机制**：普通字段是「覆盖」，`Annotated[list, operator.add]` 这类带 reducer 的字段是「累加」，所以轨迹能跨节点自动收集。
4. **基座模型也能做图节点**：分类用 few-shot 补全，创作用 one-shot 示范，统一贪心解码（可复现）+ 重复惩罚 + `<|im_end|>` 停止符，再加规则/正则兜底，1.8B 小模型就能稳定支撑多节点协作。

后续可以继续尝试：把笑话/故事/科普节点**并行 fan-out**、给路由加第 4 个分支（如「闲聊」）、或用 checkpointer 保存对话状态。